# CallHub — Assignment 3: Module A
## Transaction Management, ACID Validation & Crash Recovery


**GitHub:** https://github.com/Sanjay18255/Assignment_2_Module_A  
**Video:** 

---

## Overview

This notebook extends the Assignment 2 B+ Tree DBMS engine with:

- **Atomicity** — every transaction either completes fully or is rolled back entirely
- **Consistency** — B+ Tree index always matches stored records
- **Durability** — committed data persists via Write-Ahead Log (WAL)
- **Crash Recovery** — on restart, uncommitted transactions are automatically undone

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
if os.path.exists('callhub_wal.log'): os.remove('callhub_wal.log')

from database import DatabaseManager, WALLogger, crash_recovery, check_consistency

print('All imports OK')

## 2. Create Database and Tables

In [ ]:
db = DatabaseManager()
db.create_database('CallHub')

member_schema = {
    'member_id':     int,
    'member_name':   str,
    'iit_email':     str,
    'primary_phone': str,
    'department_id': int,
    'is_at_campus':  bool,
    'join_date':     str
}

db.create_table('CallHub', 'Member', member_schema, order=6, search_key='member_id')

member_table, _ = db.get_table('CallHub', 'Member')
print(f'Database created: {db.list_databases()}')
print(f'Tables: {db.list_tables("CallHub")[0]}')

---
## 3. Test 1 — Atomicity: Successful Transaction (Commit)

In [ ]:
print('='*60)
print('TEST 1: Successful Transaction — all ops commit together')
print('='*60)

txn1 = db.begin_transaction('CallHub')

txn1.insert('Member', {
    'member_id':1, 'member_name':'Amit Sharma', 'iit_email':'amit@org.in',
    'primary_phone':'9000000001', 'department_id':1, 'is_at_campus':True, 'join_date':'2022-07-01'
})
txn1.insert('Member', {
    'member_id':2, 'member_name':'Priya Verma', 'iit_email':'priya@org.in',
    'primary_phone':'9000000002', 'department_id':2, 'is_at_campus':True, 'join_date':'2021-07-01'
})
txn1.insert('Member', {
    'member_id':3, 'member_name':'Rahul Mehta', 'iit_email':'rahul@org.in',
    'primary_phone':'9000000003', 'department_id':3, 'is_at_campus':True, 'join_date':'2023-01-01'
})
txn1.insert('Member', {
    'member_id':4, 'member_name':'Sneha Iyer', 'iit_email':'sneha@org.in',
    'primary_phone':'9000000004', 'department_id':4, 'is_at_campus':True, 'join_date':'2020-07-01'
})
txn1.insert('Member', {
    'member_id':5, 'member_name':'Karan Patel', 'iit_email':'karan@org.in',
    'primary_phone':'9000000005', 'department_id':5, 'is_at_campus':True, 'join_date':'2019-07-01'
})

txn1.commit()

print(f'\nRecords after commit: {member_table.count()}')
print('All records:')
for k, v in member_table.get_all():
    print(f'  [{k}] {v["member_name"]:15s} | {v["primary_phone"]} | dept={v["department_id"]}')

---
## 4. Test 2 — Atomicity: Rollback on Failure

In [ ]:
print('='*60)
print('TEST 2: Rollback — failure mid-transaction, all ops undone')
print('='*60)

print(f'\nRecords BEFORE failed transaction: {member_table.count()}')
print(f'Member 1 phone BEFORE: {member_table.get(1)["primary_phone"]}')

txn2 = db.begin_transaction('CallHub')
try:
    # Op 1: Insert new member
    txn2.insert('Member', {
        'member_id':6, 'member_name':'Ghost User', 'iit_email':'ghost@org.in',
        'primary_phone':'0000000000', 'department_id':1, 'is_at_campus':False, 'join_date':'2026-01-01'
    })
    # Op 2: Update existing member
    txn2.update('Member', 1, {'primary_phone': '9999999999'})

    # Simulate a crash/failure
    raise Exception('💥 Simulated system crash mid-transaction!')

    txn2.commit()  # Never reached

except Exception as e:
    print(f'\n  Exception: {e}')
    txn2.rollback()

print(f'\nRecords AFTER rollback: {member_table.count()}')
print(f'Member 1 phone AFTER (should be original): {member_table.get(1)["primary_phone"]}')
print(f'Ghost user exists: {member_table.get(6)}')

assert member_table.count() == 5, 'Rollback failed — count is wrong!'
assert member_table.get(1)['primary_phone'] == '9000000001', 'Rollback failed — phone not restored!'
assert member_table.get(6) is None, 'Rollback failed — ghost record still exists!'
print('\n Atomicity verified — all changes undone correctly')

---
## 5. Test 3 — Atomicity: Rollback on Delete

In [ ]:
print('='*60)
print('TEST 3: Rollback of DELETE — record must be restored')
print('='*60)

print(f'Member 5 before: {member_table.get(5)["member_name"]}')

txn3 = db.begin_transaction('CallHub')
try:
    txn3.delete('Member', 5)
    print(f'After delete inside txn: {member_table.get(5)}')
    raise Exception('💥 Failure after delete!')
except Exception as e:
    print(f'  Error: {e}')
    txn3.rollback()

print(f'Member 5 after rollback: {member_table.get(5)["member_name"]}')
assert member_table.get(5) is not None
print('Delete rollback works correctly')

---
## 6. Test 4 — Consistency Check

In [ ]:
print('='*60)
print('TEST 4: Consistency — B+ Tree index matches stored records')
print('='*60)

ok, issues = db.check_consistency()
print(f'\nConsistency check result: {"PASSED" if ok else "FAILED"}')
if issues:
    for issue in issues:
        print(f'  Issue: {issue}')
else:
    print('  No inconsistencies found.')
    print('  Every key in the B+ Tree matches its record.')
    print('  Every record is searchable by its key.')

print(f'\nCurrent state of Member table:')
for k, v in member_table.get_all():
    assert member_table.get(k) is not None
    print(f'  [{k}] {v["member_name"]:15s} ✓ searchable')

---
## 7. Test 5 — Durability: WAL Log Contents

In [ ]:
import json

print('='*60)
print('TEST 5: Durability — WAL Log shows all transaction history')
print('='*60)

wal = WALLogger('callhub_wal.log')
entries = wal.read_all()

print(f'\nTotal WAL entries: {len(entries)}')
print()

# Show summary
from collections import defaultdict
txn_summary = defaultdict(list)
for e in entries:
    txn_summary[e.get('txn_id')].append(e.get('op','?'))

print(f'{"TXN":>6} {"Operations":40} {"Final Status"}')
print('-'*65)
for tid, ops in sorted(txn_summary.items()):
    status = [o for o in ops if o in ('COMMITTED','ROLLED_BACK')]
    status_str = status[0] if status else 'UNKNOWN'
    ops_str = ' → '.join(o for o in ops if o not in ('COMMITTED','ROLLED_BACK','BEGIN'))
    print(f'{str(tid):>6} {ops_str:40} {status_str}')

print(f'\nWAL provides complete audit trail of all transactions')

---
## 8. Test 6 — Crash Recovery

In [ ]:
print('='*60)
print('TEST 6: Crash Recovery — simulate crash, recover on restart')
print('='*60)

# Step 1: Simulate a crash by writing a partial transaction to WAL
print('\nStep 1: Simulating crash...')
print('  → Writing BEGIN and INSERT to WAL (no COMMIT)')
wal = WALLogger('callhub_wal.log')
wal.write({'txn_id': 9999, 'op': 'BEGIN', 'status': 'BEGIN', 'ts': '2026-04-05T10:00:00'})
wal.write({
    'txn_id': 9999, 'op': 'INSERT', 'table': 'Member', 'key': 99,
    'before': None,
    'after': {'member_id':99,'member_name':'Crash User','iit_email':'crash@org.in',
              'primary_phone':'0000000099','department_id':1,'is_at_campus':False,'join_date':'2026-01-01'},
    'status': 'PENDING', 'ts': '2026-04-05T10:00:01'
})

# Step 2: The partial data was applied to the DB before crash
member_table.insert({'member_id':99,'member_name':'Crash User','iit_email':'crash@org.in',
                     'primary_phone':'0000000099','department_id':1,'is_at_campus':False,'join_date':'2026-01-01'})
print(f'  → Crash record inserted. Count before recovery: {member_table.count()}')
print(f'  → Crash User exists: {member_table.get(99)["member_name"]}')

# Step 3: System restarts — run crash recovery
print('\nStep 2: System restarting — running crash recovery...')
db.recover('callhub_wal.log')

# Step 4: Verify recovery
print(f'\nStep 3: Verification after recovery')
print(f'  Count after recovery: {member_table.count()}')
print(f'  Crash User exists: {member_table.get(99)}')

assert member_table.get(99) is None, 'Recovery failed — crash record still exists!'
assert member_table.count() == 5
print('\n Crash recovery verified — uncommitted data removed')

---
## 9. Test 7 — Durability: Data Survives After Commit

In [ ]:
print('='*60)
print('TEST 7: Durability — committed data survives restart')
print('='*60)

# Commit a new record
txn_dur = db.begin_transaction('CallHub')
txn_dur.insert('Member', {
    'member_id':10, 'member_name':'Durable Record', 'iit_email':'dur@org.in',
    'primary_phone':'9876543210', 'department_id':1, 'is_at_campus':True, 'join_date':'2026-04-05'
})
txn_dur.commit()
print(f'Committed member 10: {member_table.get(10)["member_name"]}')

# Now simulate a restart and recovery — committed data must survive
print('\nSimulating restart and recovery...')
db.recover('callhub_wal.log')

print(f'Member 10 after recovery: {member_table.get(10)["member_name"]}')
assert member_table.get(10) is not None
assert member_table.get(10)['member_name'] == 'Durable Record'
print('Durability verified — committed data intact after recovery')

---
## 10. Final State & Summary

In [ ]:
print('='*60)
print('FINAL STATE — Member Table')
print('='*60)
for k, v in member_table.get_all():
    print(f'  [{k:2d}] {v["member_name"]:20s} | {v["primary_phone"]} | dept={v["department_id"]}')

print(f'\nTotal records: {member_table.count()}')
print(f'Tree height:   {member_table.tree_height()}')

# Final consistency check
ok, issues = db.check_consistency()
print(f'Consistency:   {"PASSED" if ok else "FAILED"}')

print()
print('='*60)
print('ACID PROPERTIES SUMMARY')
print('='*60)
print('Atomicity   — transactions commit fully or rollback fully')
print('Consistency — B+ Tree index always matches stored records')
print('Durability  — WAL ensures committed data survives crashes')
print('Recovery    — uncommitted transactions auto-rolled back on restart')

---
## 11. Conclusion

This module extended the Assignment 2 B+ Tree DBMS with full ACID transaction support:

- **Atomicity** was demonstrated by rolling back both INSERT and DELETE operations on failure, restoring the database to its exact pre-transaction state.
- **Consistency** was verified by checking that every key in the B+ Tree index matches its stored record and is searchable.
- **Durability** was implemented using a Write-Ahead Log (WAL) — every operation is logged before being applied, so committed data survives crashes.
- **Crash Recovery** was demonstrated by simulating a partial transaction (BEGIN + INSERT, no COMMIT), then restarting and automatically undoing the uncommitted changes.

**Limitations:** Isolation (multi-user concurrency) is covered in Module B. The current implementation is single-threaded.